# Intelligent Legal Document Analyzer
Run in Google Colab: upload this project folder (or `git clone` your repo), then run the cells top to bottom.

In [ ]:
!pip -q install spacy nltk scikit-learn pandas flask pypdf python-docx joblib
!python -m spacy download en_core_web_sm
# Optional abstractive summaries:
# !pip -q install transformers torch

## 1. Dataset (Pandas)

In [ ]:
import pandas as pd
df = pd.read_csv('data/legal_clauses.csv')
print(df.shape)
df['label'].value_counts()

## 2. Clause classifier (TF-IDF + Logistic Regression)

In [ ]:
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import classification_report
from analyzer.pipeline import ClauseClassifier
pred = cross_val_predict(ClauseClassifier.build(), df['text'], df['label'], cv=StratifiedKFold(4, shuffle=True, random_state=42))
print(classification_report(df['label'], pred, zero_division=0))

## 3. Full pipeline on the sample contract

In [ ]:
from analyzer import LegalAnalyzer
A = LegalAnalyzer()
text = open('data/sample_contract.txt', encoding='utf-8').read()
r = A.analyze(text)
print('Risk score:', r['score'])
r['ents']

In [ ]:
pd.DataFrame(r['clauses'])[['k','conf','s']]

In [ ]:
pd.DataFrame(r['risks'])[['label','sev','why']]

In [ ]:
print('\n'.join(r['summary']))

## 4. Abstractive summary (Hugging Face)

In [ ]:
from transformers import pipeline
summ = pipeline('summarization', model='sshleifer/distilbart-cnn-12-6')
print(summ(' '.join(text.split()[:700]), max_length=90, min_length=30, do_sample=False)[0]['summary_text'])

## 5. spaCy entity view

In [ ]:
import spacy
from spacy import displacy
nlp = spacy.load('en_core_web_sm')
displacy.render(nlp(text[:900]), style='ent', jupyter=True)